# D212 - Task 3: Market Basket Analysis
Desiree McElroy | WGU D212 Data Mining II

## Part I: Research Question

**A1: Research Question**

What prescription combinations are most frequently co-prescribed together among patients in this data set hospital network, and can those associations inform pharmacists and medical care teams regarding predictable prescription patterns?

**A2: Goal**

The goal of this analysis is to identify the top prescription item associations using the Apriori algorithm, which measures support, confidence, and lift, in order to reveal co-prescription patterns that may provide support for clinical decision making and pharmacy planning.

## Part II: Market Basket Justification

**B1: Explanation**

Market basket analysis examines the provided medical_market_basket.csv dataset by treating each patient's prescription history as a transaction. Each row in the data set represents one occurrence of a patient and their prescribed medications.

The Apriori algorithm scans all 7,501 transactions (post data processing) to find itemsets that appear together with a frequency above a minimum support threshold. It then generates association rules (for example, "patients prescribed Drug A are also frequently prescribed Drug B and Drug C") and scores each rule using the metrics: support, confidence, and lift. The expected outcome is a ranked table of rules that reveals which prescription combinations co-occur at meaningful rates across the patient data set, alongside the metrics mentioned previously.

**B2: Example transaction**

One example transaction from this dataset is a patient (first observation) whose prescription history includes: amlodipine, albuterol aerosol, allopurinol, pantoprazole, lorazepam, omeprazole, mometasone, fluconazole, gabapentin, pravastatin, cialis, losartan, metoprolol succinate XL, sulfamethoxazole, abilify, spironolactone, albuterol HFA, levofloxacin, promethazine, and glipizide. Each patient row is treated as one transaction, and each non-null prescription is treated as one item in that transaction. This specific patient had 20 active prescriptions.

**B3: Assumption**

A core assumption of market basket analysis is that item order within a transaction is irrelevant. The algorithm treats each transaction as an unordered set. In this dataset, that means the column a prescription appears in (Presc01 vs. Presc15) carries no meaning. What matters is whether two drugs co-occur in the same patient's history, not which was prescribed first. This is sometimes called the "bag of items" assumption.

There is also an assumption that each transaction is independent. The algorithm treats each patient's prescription history in isolation, with no awareness that patients may share a doctor or diagnosis within a certain region. In practice, co-prescription patterns can be driven by a specific provider, which could inflate certain association rules beyond what's clinically representative of the broader population.

## Part III: Data Preparation and Analysis

In [1]:
## imports
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns


from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

pd.set_option('display.max_rows', None)

#### Data Acquire & Explore

In [2]:
df = pd.read_csv('medical_market_basket.csv')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15002 entries, 0 to 15001
Data columns (total 20 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   Presc01  7501 non-null   object
 1   Presc02  5747 non-null   object
 2   Presc03  4389 non-null   object
 3   Presc04  3345 non-null   object
 4   Presc05  2529 non-null   object
 5   Presc06  1864 non-null   object
 6   Presc07  1369 non-null   object
 7   Presc08  981 non-null    object
 8   Presc09  654 non-null    object
 9   Presc10  395 non-null    object
 10  Presc11  256 non-null    object
 11  Presc12  154 non-null    object
 12  Presc13  87 non-null     object
 13  Presc14  47 non-null     object
 14  Presc15  25 non-null     object
 15  Presc16  8 non-null      object
 16  Presc17  4 non-null      object
 17  Presc18  4 non-null      object
 18  Presc19  3 non-null      object
 19  Presc20  1 non-null      object
dtypes: object(20)
memory usage: 2.3+ MB


In [3]:
## get a view of dataframe
df.head(10)

,Presc01,Presc02,Presc03,Presc04,Presc05,Presc06,Presc07,Presc08,Presc09,Presc10,Presc11,Presc12,Presc13,Presc14,Presc15,Presc16,Presc17,Presc18,Presc19,Presc20
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,amlodipine,albuterol aerosol,allopurinol,pantoprazole,lorazepam,omeprazole,mometasone,fluconozole,gabapentin,pravastatin,cialis,losartan,metoprolol succinate XL,sulfamethoxazole,abilify,spironolactone,albuterol HFA,levofloxacin,promethazine,glipizide
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,citalopram,benicar,amphetamine salt combo xr,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,enalapril,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,paroxetine,allopurinol,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,abilify,atorvastatin,folic acid,naproxen,losartan,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Quite a few rows with many null values. This is to be expected. Definitely worth checking if some rows have **all null values**, in which case they should be removed.

In [4]:
# how many rows are completely empty?
print(f"Total rows: {df.shape[0]}")
print(f"Rows with all null values: {df.isnull().all(axis=1).sum()}")

print(f"Rows remaining after dropping: {df.notna().any(axis=1).sum()}")
print('-------------')
print(f'Displaying Dataframe with NaN across board.')
df[df.isnull().all(axis=1)].head()

Total rows: 15002
Rows with all null values: 7501
Rows remaining after dropping: 7501
-------------
Displaying Dataframe with NaN across board.


,Presc01,Presc02,Presc03,Presc04,Presc05,Presc06,Presc07,Presc08,Presc09,Presc10,Presc11,Presc12,Presc13,Presc14,Presc15,Presc16,Presc17,Presc18,Presc19,Presc20
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Below, printing a view of the way the apriori algorithm wants the data preprocessed.

In [5]:
## viewing prep for algorithm
df.apply(lambda row: row.dropna().tolist(), axis=1).tolist()[:2]

[[],
 ['amlodipine',
  'albuterol aerosol',
  'allopurinol',
  'pantoprazole',
  'lorazepam',
  'omeprazole',
  'mometasone',
  'fluconozole',
  'gabapentin',
  'pravastatin',
  'cialis',
  'losartan',
  'metoprolol succinate XL',
  'sulfamethoxazole',
  'abilify',
  'spironolactone',
  'albuterol HFA',
  'levofloxacin',
  'promethazine',
  'glipizide']]

Below, I wanted to look at is the individual prescription names and sort them. This data set may be clean, but as due diligence, it is best to view all of these names in order to identify any possible misspellings.

In [6]:
## wanting to look at individual names to identify any possible misspellings
all_items = pd.Series(df.values.flatten()).dropna()
item_counts = all_items.value_counts().sort_index()
print(f"Total unique prescriptions: {len(item_counts)}")
print('Prescription Names')
item_counts.sort_index(ascending=True)

Total unique prescriptions: 119
Prescription Names


Duloxetine                      90
Premarin                       351
Yaz                             32
abilify                       1788
acetaminophen                  118
actonel                         90
albuterol HFA                   67
albuterol aerosol              153
alendronate                     36
allopurinol                    250
alprazolam                     595
amitriptyline                   34
amlodipine                     536
amoxicillin                     65
amphetamine                    226
amphetamine salt combo         513
amphetamine salt combo xr     1348
atenolol                        81
atorvastatin                   972
azithromycin                   107
benazepril                      69
benicar                        157
boniva                          46
bupropion sr                    31
carisoprodol                    86
carvedilol                    1306
cefdinir                        14
celebrex                        82
celecoxib           

### C1. Data Transformation

The raw dataset contained 7,501 rows and 20 columns (Presc01–Presc20), with prescription names stored as string values. Several rows were entirely null, representing patients with no recorded prescription history. These were removed, keeping only rows with at least one non-null value, resulting in a clean transaction dataset.

All prescription strings were converted to lowercase to eliminate case duplicates (e.g., "Abilify" vs. "abilify" being treated as separate items). Each row was then converted to a list of non-null prescription values, producing a list of lists where each inner list represents one patient's transaction.

This list of transactions was passed through `TransactionEncoder` from the mlxtend library, which one-hot encodes the data into a boolean dataframe. Each column represents a unique drug, and each row represents a patient, with `True` indicating the drug was prescribed and `False` indicating it was not. This format is required as input for the Apriori algorithm. A copy of the cleaned dataset is included as `cleaned_df_basket_analysis.csv`.

### C2. Apriori Algorithm

The Apriori algorithm was executed using `mlxtend.frequent_patterns.apriori` with a minimum support threshold of 0.01 (1%), meaning only drug combinations appearing in at least 1% of patient transactions were retained as frequent itemsets. Association rules were then generated using `mlxtend.frequent_patterns.association_rules` with a minimum lift threshold of 1.0 to filter out rules with no positive association. The resulting rules table was sorted by lift and confidence.

The complete code execution for this section is shown below, starting with the data preprocessing pipeline and concluding with the `apriori()` and `association_rules()` function calls and their outputs.

#### Code Execution

Steps performed in the cells below:

1. Lowercase all entries to eliminate case sensitive duplicates.
2. Remove rows where all 20 columns are null (no prescription history recorded).
3. Save the cleaned dataset to CSV.
4. Convert each row to a list of non-null items to build a list-of-lists for `TransactionEncoder`.
5. One-hot encode transactions using `TransactionEncoder` from `mlxtend`.
6. Run `apriori()` to identify frequent itemsets at minimum support = 0.01.
7. Run `association_rules()` to generate rules at minimum lift = 1.0.

In [7]:
## make all entries lower case to avoid issues with case sensitivity when viewing
df = df.apply(lambda col: col.str.lower())
## remove rows where all entries are null
cleaned_df = df[df.notna().any(axis=1)]

print(f"Original dataset: {df.shape[0]} rows")
print(f"Rows removed (all null): {df.shape[0] - cleaned_df.shape[0]}")
print(f"Cleaned dataset: {cleaned_df.shape[0]} rows, {cleaned_df.shape[1]} columns")

Original dataset: 15002 rows
Rows removed (all null): 7501
Cleaned dataset: 7501 rows, 20 columns


In [8]:
## adding arg for re-running notebook without overwriting csv
from pathlib import Path
save_csv = True
if save_csv:
    cleaned_df.to_csv(Path('/Users/desireemcelroy/WGU/WGU_MSDA/D212-Data_Mining_II/task_3/cleaned_df_basket_analysis.csv'), index=False)

In [9]:
# convert each row to a list of prescription values, creating a list of list for transaction encoder
transactions = cleaned_df.apply(lambda row: row.dropna().tolist(), axis=1).tolist()

# wanting to spot check
print(f"Number of transactions: {len(transactions)}")
print(f"Sample transaction: {transactions[:5]}")

Number of transactions: 7501
Sample transaction: [['amlodipine', 'albuterol aerosol', 'allopurinol', 'pantoprazole', 'lorazepam', 'omeprazole', 'mometasone', 'fluconozole', 'gabapentin', 'pravastatin', 'cialis', 'losartan', 'metoprolol succinate xl', 'sulfamethoxazole', 'abilify', 'spironolactone', 'albuterol hfa', 'levofloxacin', 'promethazine', 'glipizide'], ['citalopram', 'benicar', 'amphetamine salt combo xr'], ['enalapril'], ['paroxetine', 'allopurinol'], ['abilify', 'atorvastatin', 'folic acid', 'naproxen', 'losartan']]


In [10]:
te = TransactionEncoder()
te_array = te.fit(transactions).transform(transactions)
te_array[0]

array([ True, False, False,  True,  True, False,  True, False, False,
        True, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
        True, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False,  True, False,
       False, False, False, False,  True, False,  True, False, False,
       False, False, False, False, False, False,  True, False, False,
        True,  True, False, False, False, False, False, False,  True,
       False,  True, False,  True, False,  True, False, False, False,
        True, False, False, False,  True, False, False, False, False,
       False, False,  True,  True, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False])

In [11]:
te_df = pd.DataFrame(data=te_array, columns=te.columns_ )
te_df.head()

,abilify,acetaminophen,actonel,albuterol aerosol,albuterol hfa,alendronate,allopurinol,alprazolam,amitriptyline,amlodipine,...,triamcinolone ace topical,triamterene,trimethoprim ds,valaciclovir,valsartan,venlafaxine xr,verapamil sr,viagra,yaz,zolpidem
0,True,False,False,True,True,False,True,False,False,True,...,False,False,False,False,False,False,False,False,False,False
1,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
3,False,False,False,False,False,False,True,False,False,False,...,False,False,False,False,False,False,False,False,False,False
4,True,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


#### Step 6: Run Apriori Algorithm - Frequent Itemsets

The cell below executes `apriori()` on the one-hot encoded transaction dataframe. It returns a dataframe of frequent itemsets and their support values, filtered to sets appearing in at least 1% of transactions.

In [12]:
# Find frequent itemsets - min_support is the minimum frequency threshold
frequent_itemsets = apriori(te_df, min_support=0.01, use_colnames=True)
frequent_itemsets.sort_values('support', ascending=False).head(15)

,support,itemsets
0,0.238368,(abilify)
9,0.179709,(amphetamine salt combo xr)
15,0.174110,(carvedilol)
38,0.170911,(glyburide)
25,0.163845,(diazepam)
47,0.132116,(losartan)
11,0.129583,(atorvastatin)
46,0.098254,(lisinopril)
52,0.095321,(metoprolol)
27,0.095054,(doxycycline hyclate)


**Takeaway**

23.8% of the patient transactions in the dataset include abilify, roughly 1,785 out of 7,501 patients. 
That's a fairly high single-item support, which means abilify will likely show up in a lot of association rules.

#### Step 7: Generate Association Rules

The cell below executes `association_rules()` on the frequent itemsets to produce a full rules table. Rules are filtered to lift >= 1.0 and sorted by lift then confidence. This output confirms error-free execution and produces the `rules` dataframe used in C3 and C4.

In [13]:
# Generate association rules from frequent itemsets
# min_threshold applies to lift, keeping rules with lift >= 1.0
rules = association_rules(frequent_itemsets, metric='lift', min_threshold=1.0)
rules = rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].sort_values(['lift','confidence'], ascending=False)

rules.head(10)

,antecedents,consequents,support,confidence,lift
299,(methylprednisone),(lisinopril),0.015998,0.323450,3.291994
298,(lisinopril),(methylprednisone),0.015998,0.162822,3.291994
376,"(abilify, carvedilol)",(lisinopril),0.017064,0.285714,2.907928
381,(lisinopril),"(abilify, carvedilol)",0.017064,0.173677,2.907928
366,"(abilify, carvedilol)",(glipizide),0.010265,0.171875,2.609786
371,(glipizide),"(abilify, carvedilol)",0.010265,0.155870,2.609786
91,(amphetamine salt combo),(metoprolol),0.016131,0.235867,2.474464
90,(metoprolol),(amphetamine salt combo),0.016131,0.169231,2.474464
76,(amlodipine),(metoprolol),0.016664,0.233209,2.446574
77,(metoprolol),(amlodipine),0.016664,0.174825,2.446574


### C3. Association Rules Table

The association rules were generated in C2 above. The full rules table is exported as `market_basket_rules.csv`. The table below shows the top 10 rules sorted by lift, then confidence, generated with a minimum support of 0.01 and minimum lift of 1.0. The code cell below re-displays the top results for reference.

| Antecedents | Consequents | Support | Confidence | Lift |
|---|---|---|---|---|
| methylprednisone | lisinopril | 0.0160 | 0.3235 | 3.292 |
| lisinopril | methylprednisone | 0.0160 | 0.1628 | 3.292 |
| abilify, carvedilol | lisinopril | 0.0171 | 0.2857 | 2.908 |
| lisinopril | abilify, carvedilol | 0.0171 | 0.1737 | 2.908 |
| abilify, carvedilol | glipizide | 0.0103 | 0.1719 | 2.610 |
| glipizide | abilify, carvedilol | 0.0103 | 0.1559 | 2.610 |
| amphetamine salt combo | metoprolol | 0.0161 | 0.2359 | 2.474 |
| metoprolol | amphetamine salt combo | 0.0161 | 0.1692 | 2.474 |
| amlodipine | metoprolol | 0.0167 | 0.2332 | 2.447 |
| metoprolol | amlodipine | 0.0167 | 0.1748 | 2.447 |

### C4. Top Three Rules

The top three rules were selected based on the highest lift values, with confidence used as a tiebreaker. All three involve cardiovascular medications co-occurring with other drugs.

**Rule 1: methylprednisone → lisinopril**
Support: 0.016 | Confidence: 0.323 | Lift: 3.292

This is the strongest rule in the dataset. Patients prescribed methylprednisone (a corticosteroid) are 3.29 times more likely to also be prescribed lisinopril (an ACE inhibitor) than would be expected by chance (MedlinePlus, 2023). The confidence of 0.323 means that roughly 1 in 3 patients on methylprednisone are also on lisinopril. This association is clinically grounded as corticosteroids are known to elevate blood pressure, and lisinopril is a commonly used antihypertensive. The pattern likely reflects a prescribing protocol where providers proactively manage medication induced hypertension.

**Rule 2: {abilify, carvedilol} → lisinopril**
Support: 0.017 | Confidence: 0.286 | Lift: 2.908

Patients co-prescribed abilify (an antipsychotic) and carvedilol (a beta-blocker) are 2.91 times more likely to also be on lisinopril (MedlinePlus, 2023). The confidence of 0.286 means roughly 29% of patients on both abilify and carvedilol also have lisinopril in their history. This could suggest a subset of patients managing both psychiatric conditions and cardiovascular disease, where different prescribers may be involved. The presence of all three drugs in the same patient's history shows a complex, multi system medicate regimen.

**Rule 3: {abilify, carvedilol} → glipizide**
Support: 0.010 | Confidence: 0.172 | Lift: 2.610

Patients co-prescribed abilify (an antipsychotic) and carvedilol (a beta-blocker) are 2.61 times more likely to also be prescribed glipizide, an oral medication used to manage type 2 diabetes (MedlinePlus, 2023). This rule has the third highest lift in the dataset. The confidence of 0.172 means roughly 17% of patients on both abilify and carvedilol also have glipizide in their history. The combination clearly shows a patient profile managing mental health conditions, cardiovascular disease, and diabetes simultaneously. This is a medically complex cohort that likely needs more coordinated specialty care.

## Part IV: Data Summary and Implications

**D1: Significance of Support, Lift, and Confidence**

Across the rules generated, support values ranged from 0.010 to 0.039, meaning the strongest rules apply to roughly 1% to 4% of the patient population per rule. While individual support values are not high, this is expected in a medical prescription dataset where drug combinations are condition specific. Lift values ranged from 1.0 to 3.29, with the highest lift rules concentrated around cardiovascular and psychiatric drug pairings. Any lift above 1.0 indicates a positive association beyond random co-occurrence; the top rules, clustering around 2.4–3.3, represent meaningfully elevated co-prescription rates. Confidence values for the top rules ranged from approximately 0.16 to 0.42. Rule 2 ({abilify, carvedilol} → lisinopril at 0.286) shows that nearly 1 in 3 patients on both an antipsychotic and a beta-blocker are also on an ACE inhibitor, a pattern consistent with managing psychiatric and cardiovascular conditions concurrently. Together, these three metrics confirm that the top rules are statistically significant, reflecting real and consistent prescribing patterns in this data set.

**D2: Practical Significance of Findings**

The top rules consistently point to a patient group with both cardiovascular and psychiatric diagnoses. Lisinopril, carvedilol, and abilify appear together across multiple rules, which suggests this comorbidity profile is not a statistical anomaly, but a real pattern in this population. In practice, these findings could support pharmacy systems that flag expected co-prescriptions for reconciliation reviews, or prompt care coordinators to cross-check cardiovascular screening for patients already on psychiatric medications. The support values are low, so each rule applies to a small fraction of the total patient population. Whether that reflects a clinically distinct subgroup or prescribing patterns concentrated among a handful of providers is a question this dataset alone cannot answer, and may be worth investigating with more granular data.

**D3: Recommendation**

Based on the research question and the results of this analysis, the hospital network should consider implementing co-prescription awareness flags in the pharmacy management system for the drug combinations identified in the top rules. Specifically, patients prescribed methylprednisone should be systematically reviewed for blood pressure management needs, given the strong association with lisinopril (lift 3.29). Additionally, care teams treating patients on abilify should be alerted to the elevated likelihood of concurrent cardiovascular medication use, as multiple top rules involve abilify co-occurring with carvedilol and lisinopril. This does not imply these combinations are inappropriate. Rather, it provides a data-driven basis for coordinating care across specialties and reducing the risk of medication oversights in medically complex patients.

---

## References

IBM. (2026). *What is the Apriori algorithm?* 

https://www.ibm.com/think/topics/apriori-algorithm


Khanal, B. (n.d.). *A deep dive into the Apriori algorithm for association rule mining*. Medium.

https://medium.com/@bishal.khanal.2057/a-deep-dive-into-the-apriori-algorithm-for-association-rule-mining-0e10c032354f


MCP Analytics. (n.d.). *Association rules & Apriori: A practical guide for data-driven decisions*.

https://mcpanalytics.ai/articles/association-rules-apriori-practical-guide-for-data-driven-decisions


MedlinePlus. (2023). *Lisinopril*. U.S. National Library of Medicine.

https://medlineplus.gov/druginformation.html


Rasbt. (2023). *mlxtend: Association rules*.

http://rasbt.github.io/mlxtend/user_guide/frequent_patterns/association_rules/


Rasbt. (2023). *mlxtend: TransactionEncoder*.

http://rasbt.github.io/mlxtend/user_guide/preprocessing/TransactionEncoder/


Smartbridge. (2022). *Market basket analysis 101*.

https://smartbridge.com/market-basket-analysis-101/


Wikipedia. (n.d.). *Apriori algorithm*. Wikipedia.

https://en.wikipedia.org/wiki/Apriori_algorithm